In [9]:
pip install SPARQLWrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 9.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [SPARQLWrapper]
Note: you may need to restart the kernel to use updated packages.


Scrape wiki for name of 2024 olympic swimmers and their main links. 

In [6]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re

# load page
url = "https://en.wikipedia.org/wiki/Swimming_at_the_2024_Summer_Olympics"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
soup = BeautifulSoup(html, "html.parser")

# extract athelete names
base = "https://en.wikipedia.org"
athletes = set()

for a in soup.select("a[href^='/wiki/']"):
    href = a.get("href")

    if (
        href
        and not ":" in href               
        and not href.endswith(".jpg")
        and not href.endswith(".png")
    ):
        name = href.split("/wiki/")[-1]

        # match to naming convention
        if re.match(r"^[A-Za-z0-9_\-]+$", name):
            athletes.add(urljoin(base, href))

athletes = list(athletes)

print("Total extracted links:", len(athletes))

# create dataset of links
data = []

for link in athletes[:100]:  
    name = link.split("/wiki/")[-1].replace("_", " ")

    data.append({
        "athlete": name,
        "wiki_link": link
    })

df = pd.DataFrame(data)

# save athelete and links
df.to_csv("swimmers_from_hyperlinks.csv", index=False)

print(df.head())

Total extracted links: 250
                                 athlete  \
0                          Simone Manuel   
1                         Chris Guiliano   
2  Taekwondo at the 2024 Summer Olympics   
3   200 metre backstroke at the Olympics   
4                          Olivia Wunsch   

                                           wiki_link  
0        https://en.wikipedia.org/wiki/Simone_Manuel  
1       https://en.wikipedia.org/wiki/Chris_Guiliano  
2  https://en.wikipedia.org/wiki/Taekwondo_at_the...  
3  https://en.wikipedia.org/wiki/200_metre_backst...  
4        https://en.wikipedia.org/wiki/Olivia_Wunsch  


Scrape birthday from each athlete link

In [8]:
from SPARQLWrapper import SPARQLWrapper, JSON
import time

# load data
df = pd.read_csv("swimmers_from_hyperlinks.csv")

# select date of birth of wiki convention
def get_dob_and_country(name):
    endpoint = "https://query.wikidata.org/sparql"

    query = f"""
    SELECT ?dob ?countryLabel WHERE {{
      ?athlete rdfs:label "{name}"@en;
               wdt:P569 ?dob;
               wdt:P27 ?country.

      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "en".
      }}
    }}
    LIMIT 1
    """

    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    try:
        result = sparql.query().convert()
        bindings = result["results"]["bindings"]

        if not bindings:
            return None, None

        r = bindings[0]

        dob = r["dob"]["value"]
        country = r["countryLabel"]["value"]

        return dob, country

    except:
        return None, None

# loop through all atheletes
dobs = []
countries = []

for i, row in df.iterrows():
    name = row["athlete"]

    dob, country = get_dob_and_country(name)

    dobs.append(dob)
    countries.append(country)

    time.sleep(0.3)  # avoid Wikidata rate limits

# add to dataframe
df["dob"] = dobs
df["country"] = countries

# preprocess for tableau visualization
df["dob"] = pd.to_datetime(df["dob"], errors="coerce")

df["birth_month"] = df["dob"].dt.month

df["birth_quarter"] = df["birth_month"].apply(
    lambda x: "" if pd.isna(x) else
              "Q1" if x <= 3 else
              "Q2" if x <= 6 else
              "Q3" if x <= 9 else "Q4"
)

# save file to csv
df.to_csv("swimmers_full_dataset_with_dob.csv", index=False)

print(df.head())

                                 athlete  \
0                          Simone Manuel   
1                         Chris Guiliano   
2  Taekwondo at the 2024 Summer Olympics   
3   200 metre backstroke at the Olympics   
4                          Olivia Wunsch   

                                           wiki_link  \
0        https://en.wikipedia.org/wiki/Simone_Manuel   
1       https://en.wikipedia.org/wiki/Chris_Guiliano   
2  https://en.wikipedia.org/wiki/Taekwondo_at_the...   
3  https://en.wikipedia.org/wiki/200_metre_backst...   
4        https://en.wikipedia.org/wiki/Olivia_Wunsch   

                        dob        country  birth_month birth_quarter  
0 1996-08-02 00:00:00+00:00  United States          8.0            Q3  
1 2003-06-25 00:00:00+00:00  United States          6.0            Q2  
2                       NaT           None          NaN                
3                       NaT           None          NaN                
4 2006-05-31 00:00:00+00:00      Austr

Clean data to be all lowercase to align with olympic_medals csv from kaggle and join data on "atheletes" for easy tableau visualization. 

In [9]:
import pandas as pd
import os

files = ["swimmers_full_dataset_with_dob.csv", "olympic_medals.csv"]

for file in files:
    df = pd.read_csv(file)

    # lowercase all values for athletes
    df.columns = df.columns.str.lower()
    df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

    # create new filename
    base, ext = os.path.splitext(file)
    new_file = f"{base}_cleaned{ext}"

    # save to new file
    df.to_csv(new_file, index=False)

    print(f"Saved cleaned file as: {new_file}")

Saved cleaned file as: swimmers_full_dataset_with_dob_cleaned.csv
Saved cleaned file as: olympic_medals_cleaned.csv


/var/folders/9z/k_hlbg2d3kj0x3s6rfkxtvgc0000gn/T/ipykernel_54264/976788075.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
/var/folders/9z/k_hlbg2d3kj0x3s6rfkxtvgc0000gn/T/ipykernel_54264/976788075.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
